# Day 2, Segment 1: Exercise - Custom RAG Pipeline with Advanced Chunking

## Exercise Overview

In this exercise, you'll build a production-ready RAG pipeline with **custom chunking logic** that intelligently splits documents based on semantic structure rather than raw character counts.

### What You'll Learn
- ✅ Implement domain-aware document chunking
- ✅ Preserve semantic meaning through metadata
- ✅ Build modular RAG components
- ✅ Handle real-world document formatting challenges

### Business Context
You're building a product strategy tool that ingests company notes (organized by sections) and answers strategic questions. The challenge: standard chunking loses section context, making retrieval less relevant.

**Solution**: Parse sections explicitly and preserve structure as metadata.

### Exercise Structure
- **7 TODO functions** to implement
- **2 test scenarios** to validate your work
- **Evaluation checklist** at the end

Let's begin!

## Part 1: Import Dependencies & Setup

**What to do**: Review the imports. All required LangChain modules are included.

**Key imports**:
- `TextLoader`: Load documents from files
- `OpenAIEmbeddings`: Generate semantic vectors
- `Chroma`: Vector store for similarity search
- `ChatOpenAI`: LLM for answer generation

Run the cell below to import all dependencies.

In [6]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Environment configured")

✓ Environment configured


---

## TODO 1: Load Documents Function

**Objective**: Implement a function to load raw documents from a text file.

**What to do**:
1. Create a `TextLoader` pointing to `"data/notes.txt"`
2. Call `.load()` to extract documents
3. Return the list of Document objects

**Why it matters**: 
- This is the ingestion step of our RAG pipeline
- The loaded documents will be processed by subsequent functions
- In production, you might load from PDFs, web pages, databases, etc.

**Hint**: TextLoader expects a file path and returns a list of Document objects

### Your task:
Implement the `load_documents()` function below:

In [8]:
# ========================
# TODO 1: Load Documents
# ========================
def load_documents():
    """
    Load raw documents from file.
    
    Expected file format: data/notes.txt
    
    Returns:
        List of LangChain Document objects
    """
    loader = TextLoader("data/notes.txt")
    return loader.load()


# Test: Run this to verify loading works
docs = load_documents()
print(f"✓ Loaded {len(docs)} document(s)")
if docs:
    print(f"  Preview: {docs[0].page_content[:100]}...")

✓ Loaded 1 document(s)
  Preview: Pricing Strategy

Subscription pricing is widely adopted in B2B SaaS because it provides predictable...


---

## TODO 2: Custom Chunking Function

**Objective**: Implement intelligent document chunking that preserves section structure.

**Challenge**: Standard character-based chunking (like `RecursiveCharacterTextSplitter`) loses the semantic meaning of document sections. Your solution must parse sections explicitly.

**Document Format**:
```
[Section Title]
Content line 1
Content line 2

---

[Next Section]
More content
```

**What to do**:
1. For each document, split by `"\n\n---\n\n"` separator
2. For each section:
   - Extract the title (first line)
   - Get the remaining content
   - Create a new Document with:
     - `page_content`: section content (without title)
     - `metadata`: Include `"section"` (title) and `"source"` (filename)
3. Return all chunks

**Why it matters**:
- Section context improves retrieval relevance
- Metadata enables filtering and ranking
- Structured chunking scales better than character-based splits

**Example**:
```python
Input: "Pricing\nSubscription model\n\n---\n\nFeatures\nMulti-tier..."
Output: [
    Document(page_content="Subscription model", metadata={"section": "Pricing", ...}),
    Document(page_content="Multi-tier...", metadata={"section": "Features", ...})
]
```

### Your task:
Implement the `custom_chunking()` function below:

In [9]:
# ========================
# TODO 2: Custom Chunking
# ========================
def custom_chunking(docs):
    """
    Split documents by section structure, preserving semantic boundaries.
    
    Args:
        docs: List of Document objects from loader
        
    Returns:
        List of Document chunks with section metadata
    """
    chunks = []

    for doc in docs:
        # Split by section separator
        sections = doc.page_content.split("\n\n---\n\n")

        for section in sections:
            section = section.strip()

            if not section:
                continue

            # Extract section title (first line)
            lines = section.split("\n")
            title = lines[0]

            # Remaining content
            content = "\n".join(lines[1:])

            chunks.append(
                Document(
                    page_content=content,
                    metadata={
                        "section": title,
                        "source": "notes.txt"
                    }
                )
            )

    return chunks


# Test: Verify chunking
chunks = custom_chunking(docs)
print(f"✓ Created {len(chunks)} chunk(s)")
for i, chunk in enumerate(chunks[:3]):
    print(f"  Chunk {i+1}: Section '{chunk.metadata['section']}'")
    print(f"           Content: {chunk.page_content[:50]}...")

✓ Created 6 chunk(s)
  Chunk 1: Section 'Pricing Strategy'
           Content: 
Subscription pricing is widely adopted in B2B Saa...
  Chunk 2: Section 'Customer Behavior'
           Content: 
Enterprise customers prioritize reliability, secu...
  Chunk 3: Section 'Revenue Insights'
           Content: 
Companies that transitioned from freemium to tier...


---

## TODO: Create Vector Store

**Objective**: Build a vector store that indexes your chunks for semantic search.

**What to do**:
1. Initialize `OpenAIEmbeddings` with:
   - `model="text-embedding-3-small"`
   - `openai_api_key=os.getenv("OPENAI_API_KEY")`
2. Create a Chroma vector store using:
   - `Chroma.from_documents(chunks, embedding=embeddings, collection_name="rag_chunking_demo")`
3. Return the vector store

**Why it matters**:
- Embeddings convert text to semantic vectors (1536 dimensions)
- Chroma indexes these vectors for fast similarity search
- Collection names organize different document sets

**Key point**: Use the same embeddings model consistently across all components!

### Your task:
Implement the `create_vector_store()` function below:

In [ ]:
# ========================
# TODO 3: Create Vector Store
# ========================
def create_vector_store(chunks):
    """
    Create a Chroma vector store from document chunks.
    
    Args:
        chunks: List of Document chunks with embeddings
        
    Returns:
        Chroma VectorStore instance
    """
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small", 
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    return Chroma.from_documents(
        chunks,
        embedding=embeddings,
        collection_name="rag_chunking_demo"
    )


# Test: Create vector store
vector_store = create_vector_store(chunks)
print("✓ Vector store created and indexed")

---

## TODO: Retrieval Function

**Objective**: Implement semantic search to find relevant documents.

**What to do**:
1. Call `vector_store.similarity_search(query, k=3)` to find top 3 relevant chunks
2. Return the list of retrieved documents

**Why it matters**:
- Similarity search finds semantically related content
- The retrieved documents become the "ground truth" for LLM answer generation
- Retrieval quality directly impacts answer accuracy

**Key parameter**: `k=3` means retrieve 3 most similar documents (standard for RAG)

### Your task:
Implement the `retrieve_context()` function below:

In [ ]:
# ========================
# TODO 4: Retrieval
# ========================
def retrieve_context(vector_store, query):
    """
    Retrieve the most relevant documents for a query.
    
    Args:
        vector_store: Chroma VectorStore instance
        query: User query string
        
    Returns:
        List of top 3 most similar Document objects
    """
    # Retrieve top 3 relevant chunks
    return vector_store.similarity_search(query, k=3)


# Test: Retrieve context
test_query = "What pricing strategy works best for enterprise customers?"
retrieved = retrieve_context(vector_store, test_query)
print(f"✓ Retrieved {len(retrieved)} document(s)")
for i, doc in enumerate(retrieved):
    print(f"  Result {i+1}: Section '{doc.metadata['section']}'")
    print(f"           Content: {doc.page_content[:50]}...")

---

## TODO: Context Builder

**Objective**: Format retrieved documents into a cohesive context string for the LLM.

**What to do**:
1. For each retrieved document:
   - Get the section title from `metadata['section']`
   - Create a formatted string: `"[Section Title]\nContent"`
2. Join all formatted sections with `"\n\n"`
3. Return the complete context string

**Why it matters**:
- LLMs need well-formatted inputs to perform best
- Section headers provide context clues
- Clear formatting reduces confusion about content boundaries

**Example output**:
```
[Pricing]
Subscription model is most effective...

[Enterprise]
Large customers require...
```

### Your task:
Implement the `build_context()` function below:

In [ ]:
# ========================
# TODO 5: Context Builder
# ========================
def build_context(docs):
    """
    Format retrieved documents into structured context for LLM.
    
    Args:
        docs: List of retrieved Document objects
        
    Returns:
        Formatted context string with section headers
    """
    context_parts = []

    for doc in docs:
        section = doc.metadata.get("section", "Unknown")
        context_parts.append(f"[{section}]\n{doc.page_content}")

    return "\n\n".join(context_parts)


# Test: Build context
context = build_context(retrieved)
print("✓ Context formatted for LLM")
print(f"Context length: {len(context)} characters")
print("\n--- Context Preview ---")
print(context[:300] + "...")

---

## TODO: LLM Answer Generation

**Objective**: Use ChatOpenAI to generate answers grounded in retrieved context.

**What to do**:
1. Initialize `ChatOpenAI` with:
   - `model="gpt-4o"`
   - `temperature=0` (deterministic responses)
   - `openai_api_key=os.getenv("OPENAI_API_KEY")`
2. Create messages:
   - `SystemMessage`: "You are a senior product strategist."
   - `HumanMessage`: Include the context and query in a clear prompt
3. Call `llm.invoke(messages)` and return `.content`

**Why it matters**:
- System message sets the AI's role/personality
- Context injection grounds the answer in facts
- temperature=0 ensures consistent, reproducible responses

**Prompt structure**:
```
You are a senior product strategist.

Use the context below to answer the question.

Context:
[RETRIEVED_DOCUMENTS]

Question:
[USER_QUERY]
```

### Your task:
Implement the `generate_answer()` function below:

In [ ]:
# ========================
# TODO 6: LLM Response
# ========================
def generate_answer(context, query):
    """
    Generate an answer using the LLM with retrieved context.
    
    Args:
        context: Formatted context string from retrieved documents
        query: User query
        
    Returns:
        Generated answer string
    """
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    messages = [
        SystemMessage(content="You are a senior product strategist."),
        HumanMessage(content=f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content


# Test: Generate answer
answer = generate_answer(context, test_query)
print("✓ Answer generated")
print(f"\n--- Answer ---\n{answer}")

---

## TODO: Complete RAG Pipeline

**Objective**: Orchestrate all components into a single, reusable function.

**What to do**:
1. Call `load_documents()` to load raw data
2. Call `custom_chunking()` to parse sections
3. Call `create_vector_store()` to index chunks
4. Call `retrieve_context()` with the query
5. Call `build_context()` to format retrieved docs
6. Call `generate_answer()` with context and query
7. Return the final answer

**Why it matters**:
- Single entry point for the entire pipeline
- Easier to test and debug each component
- Production-ready structure for scaling

**Execution Flow**:
```
User Query
    ↓
load_documents() → custom_chunking() → create_vector_store()
    ↓
retrieve_context() → build_context() → generate_answer()
    ↓
Answer
```

### Your task:
Implement the `rag_pipeline()` function below:

In [ ]:
# ========================
# TODO 7: Full Pipeline
# ========================
def rag_pipeline(query):
    """
    Complete RAG pipeline: load → chunk → embed → retrieve → generate.
    
    Args:
        query: User query string
        
    Returns:
        Generated answer based on retrieved context
    """
    # Load documents
    docs = load_documents()

    # Parse sections
    chunks = custom_chunking(docs)

    # Create vector store
    vector_store = create_vector_store(chunks)

    # Retrieve relevant documents
    retrieved_docs = retrieve_context(vector_store, query)

    # Format context
    context = build_context(retrieved_docs)

    # Generate answer
    answer = generate_answer(context, query)

    return answer

---

## Part 2: Testing & Validation

### Test Scenario 1: Pricing Strategy

**Query**: "What pricing strategy works best for enterprise customers?"

**Expected behavior**:
1. Pipeline retrieves relevant pricing documents
2. LLM synthesizes strategy recommendations
3. Answer includes enterprise-specific insights

### Test Scenario 2: Feature Expansion

**Query**: "What new features should we add?"

**Expected behavior**:
1. Pipeline finds feature-related sections
2. LLM recommends based on context
3. Answer reflects company priorities

### Run both tests below:

In [ ]:
# ========================
# Test Scenario 1: Pricing Strategy
# ========================
print("=" * 60)
print("Test 1: Pricing Strategy Question")
print("=" * 60)

query1 = "What pricing strategy works best for enterprise customers?"
print(f"\n📝 Query: {query1}\n")

response1 = rag_pipeline(query1)
print(f"💡 Answer:\n{response1}\n")


# ========================
# Test Scenario 2: Feature Expansion
# ========================
print("=" * 60)
print("Test 2: Feature Expansion Question")
print("=" * 60)

query2 = "What new features should we add?"
print(f"\n📝 Query: {query2}\n")

response2 = rag_pipeline(query2)
print(f"💡 Answer:\n{response2}\n")

---

## Evaluation Checklist

Use this checklist to verify your implementation:

### Document Loading ✓
- [ ] `load_documents()` successfully loads `data/notes.txt`
- [ ] Loaded documents have `page_content` and `metadata` fields
- [ ] Can print preview of first document

### Custom Chunking ✓
- [ ] `custom_chunking()` splits documents by `"\n\n---\n\n"` separator
- [ ] Each chunk preserves section title in metadata
- [ ] Each chunk contains meaningful content (not empty)
- [ ] Chunk count matches expected sections

### Vector Store Creation ✓
- [ ] `create_vector_store()` initializes OpenAIEmbeddings
- [ ] Chroma vector store is created successfully
- [ ] Collection name is "rag_chunking_demo"
- [ ] No errors about API key or invalid dimensions

### Retrieval ✓
- [ ] `retrieve_context()` returns exactly 3 documents
- [ ] Retrieved documents are ranked by relevance
- [ ] Section metadata is present in results
- [ ] Content is returned as page_content, not metadata

### Context Building ✓
- [ ] `build_context()` formats sections with headers
- [ ] Format is `"[Section Title]\nContent"`
- [ ] Multiple sections are separated by `"\n\n"`
- [ ] Output is a single string suitable for LLM prompt

### Answer Generation ✓
- [ ] `generate_answer()` initializes ChatOpenAI with correct model
- [ ] System message establishes "senior product strategist" role
- [ ] Prompt includes context and query
- [ ] Returns `llm.invoke(messages).content` (string, not Message object)
- [ ] Temperature is 0 for deterministic responses

### Pipeline Orchestration ✓
- [ ] `rag_pipeline()` calls all 6 functions in correct order
- [ ] Pipeline completes without errors
- [ ] Final answer is grounded in retrieved context
- [ ] Both test queries (pricing, features) return valid answers

---

## Challenge Extensions

### Challenge 1: Add Confidence Scoring
**Difficulty**: Medium

Modify `retrieve_context()` to return documents with similarity scores. Then, in `generate_answer()`, include the scores in the system message to improve grounding.

```python
# Example modification
retrieved_docs, scores = vector_store.similarity_search_with_score(query, k=3)
# Pass scores to generate_answer and mention in system message
```

### Challenge 2: Implement Query Expansion
**Difficulty**: Medium

Before retrieval, expand the query to capture related concepts:

```python
def expand_query(query):
    """Use LLM to generate 2-3 related questions"""
    # 1. Query for "pricing strategy"
    # 2. Also query for "pricing models" and "enterprise customers"
    # Combine results and deduplicate
```

### Challenge 3: Add Metadata Filtering
**Difficulty**: Medium

Allow filtering by section before generation:

```python
def retrieve_context(vector_store, query, section=None):
    results = vector_store.similarity_search(query, k=3)
    if section:
        results = [r for r in results if r.metadata['section'] == section]
    return results
```

### Challenge 4: Implement Result Ranking
**Difficulty**: Advanced

Score retrieved documents by multiple factors:
- Semantic similarity
- Section relevance
- Content length (prefer longer, more detailed sections)

```python
def rank_results(results, query):
    """Score by similarity + section + length"""
    # Each document gets a composite score
    return sorted(results, key=lambda x: composite_score(x, query))
```

---

## Learning Outcomes

After completing this exercise, you should understand:

1. **Custom Chunking Strategy**: How to split documents intelligently based on structure
2. **Metadata Preservation**: Importance of maintaining context through metadata fields
3. **RAG Pipeline Architecture**: Complete flow from ingestion to generation
4. **Semantic Search**: How embeddings enable meaningful document retrieval
5. **Prompt Engineering**: Structuring prompts for LLM grounding

## Key Insights

| Concept | Insight |
|---------|---------|
| **Chunking** | Structure-aware chunking > character-based chunking |
| **Metadata** | Rich metadata enables better retrieval and ranking |
| **Retrieval** | Top-k retrieval is fast; filtering adds precision |
| **Context** | Well-formatted context improves LLM accuracy |
| **Testing** | Multiple query types validate pipeline robustness |

## Next Steps

In Segment 2, you'll learn:
- Multi-query retrieval expansion
- Advanced filtering and ranking
- Hybrid search combining dense and sparse methods
- Dynamic context window sizing

Great work building a production RAG pipeline! 🎉